In [93]:
import pandas as pd
import numpy as np
import re

In [94]:
data_path = "../data/WWW Bottom Mold list- All brands - 2026 (2).xlsx"

raw = pd.read_excel(data_path)

In [95]:
def clean_df_col_names(df):
    
    cols = {}
    for col in df.columns:
        new_col = str(col).split("\n")[0].strip()
        cols[col] = new_col.strip().lower().replace(' ', '_').replace('-', '_')
    df.rename(columns=cols, inplace=True)
    return df

def replace_whitespace(df):
    return df.replace(
    to_replace=[
        r"^\s*$",     # empty or whitespace-only
        r"^NA$",      # NA
        r"^N/A$",     # N/A
        r"^\xa0$",    # non-breaking space
    ],
    value=np.nan,
    regex=True
)

def normalize_season(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().upper()

    # -----------------------------------
    # Normalize season prefixes
    # -----------------------------------
    replacements = {
        "SPR": "S",
        "SS": "S",
        "AW": "F",
        "FW": "F"
    }

    for old, new in replacements.items():
        if value.startswith(old):
            value = value.replace(old, new, 1)

    # -----------------------------------
    # Validate final format
    # -----------------------------------
    pattern = r'^[SF]\d{2}$'

    if re.match(pattern, value):
        return value

    return np.nan

import numpy as np
import pandas as pd


def normalize_location(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().title()

    # -----------------------------------
    # Normalize aliases
    # -----------------------------------
    replacements = {
        "North Vietnam": "Vietnam",
        "South Vietnam": "Vietnam",
        "Viet Nam": "Vietnam",
        "Indonesia ": "Indonesia"
    }

    return replacements.get(value, value)

In [96]:
df =raw.copy()

df = clean_df_col_names(df) # Clean column names
df = replace_whitespace(df) # replace whitespaces

# Define data types for each column
dtype_set ={
    'brand': str, 
    'part_name': str, 
    'mold_code': str, 
    'compound': str, 
    'vendor_name': str,
    'location': str, 
    'mold_shop': str, 
    'season': str, 
    'a_mold_cost': float, 
    'last_demand_season': str,
    'mold_ownership': str, 
    'mold_condition': str, 
    'daily_output': float, 
    'total_mold_qty': float,
    '1': float, '1.5': float, '2': float, '2.5': float, '3': float, '3.5': float, '4': float, '4.5': float, '5': float, '5.5': float, '6': float, '6.5': float,
    '7': float, '7.5': float, '8': float, '8.5': float, '9': float, '9.5': float, '10': float, '10.5': float, '11': float, '11.5': float, '12': float,
    '17.5': float, '18': float, 
    'comments': str, 'remark': str, 'actual_output': str
}
for col, dtype in dtype_set.items():
    if col in df.columns and dtype == str:
        df[col] = df[col].astype(dtype)
    elif col in df.columns and dtype == float:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        print(f"Warning: Column '{col}' not found in DataFrame.")


# keep only valid season values in 'last_demand_season' & 'season' column
df['season'] = df['season'].apply(normalize_season)
df['last_demand_season'] = df['last_demand_season'].apply(normalize_season)

# split brand
df['brand'] = df['brand'].apply(lambda x: x.split("/"))

# normalize vendor_name
df['vendor_name'] = df['vendor_name'].str.strip().str.upper()

#normalize location
df['location'] = df['location'].apply(normalize_location)

In [97]:
df = df[((df['part_name'] =="Outsole") | (df['part_name'] =="Midsole"))]

In [104]:
import pandas as pd
import numpy as np
import re
from datetime import date

# ----------------------------
# Helpers
# ----------------------------
def none_if_nan(x):
    if pd.isna(x):
        return None
    return x

def to_int_or_none(x):
    x = none_if_nan(x)
    if x is None:
        return None
    try:
        return int(float(x))
    except Exception:
        return x

def to_float_or_none(x):
    x = none_if_nan(x)
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        return x

def normalize_code(s):
    s = none_if_nan(s)
    if s is None:
        return None
    s = str(s).strip().upper()
    s = re.sub(r"[^A-Z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or None

def component_code_from_name(component_name):
    s = none_if_nan(component_name)
    if s is None:
        return None
    s = str(s).strip()
    m = re.match(r"^([^\s(]+)", s)
    return m.group(1) if m else s

def ensure_list(x):
    x = none_if_nan(x)
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    return [str(x)]

# Size columns: "1", "1.5", ..., "18"
size_cols = [str(x).replace(".0", "") for x in np.arange(1, 18.5, 0.5)]

# ----------------------------
# v2 Container
# ----------------------------
result = {
    "schemaVersion": "2.0",
    "lastUpdated": date.today().isoformat(),
    "reference": {
        "vendors": {},
        "moldShops": {},
        "locations": {},
        "allowedSoleTypes": ["Outsole", "Midsole"]
    },
    "families": {}
}

# ----------------------------
# Build JSON
# ----------------------------
for _, row in df.iterrows():

    mold_code = none_if_nan(row.get("mold_code")) or none_if_nan(row.get("mold_code_note"))
    if mold_code is None:
        continue

    brand = row.get("brand")
    sole_type = none_if_nan(row.get("part_name"))
    component_name = none_if_nan(row.get("component_name"))
    compound = none_if_nan(row.get("compound"))

    component_code = component_code_from_name(component_name)
    if component_code is None:
        continue

    vendor_name = none_if_nan(row.get("vendor_name"))
    mold_shop_name = none_if_nan(row.get("mold_shop"))
    location = none_if_nan(row.get("location"))

    vendor_code = normalize_code(vendor_name) if vendor_name else None
    mold_shop_code = normalize_code(mold_shop_name) if mold_shop_name else None
    location_code = normalize_code(location) if location else None

    # ----------------------------
    # Reference catalogs
    # ----------------------------
    if vendor_code and vendor_code not in result["reference"]["vendors"]:
        result["reference"]["vendors"][vendor_code] = {
            "name": vendor_name,
            "notes": none_if_nan(row.get("vendor_name_note"))
        }

    if mold_shop_code and mold_shop_code not in result["reference"]["moldShops"]:
        result["reference"]["moldShops"][mold_shop_code] = {"name": mold_shop_name}

    if location_code and location_code not in result["reference"]["locations"]:
        result["reference"]["locations"][location_code] = {"name": location}

    # ----------------------------
    # Family block
    # ----------------------------
    fam = result["families"].setdefault(str(mold_code), {
        "moldCode": str(mold_code),
        "brands": [],
        "notes": None,
        "stylesUsingThisFamily": [],
        "sourcingRules": [],
        "components": {},

        # ### NEW: sizing rules live at family level (easier to export 1 sheet per family)
        "sizingRulesByComponent": {}
    })

    # ----------------------------
    # Component block
    # ----------------------------
    if sole_type is None:
        sole_type = "Unknown"

    sole_block = fam["components"].setdefault(str(sole_type), {})

    comp = sole_block.setdefault(str(component_code), {
        "displayName": component_name,
        "compound": compound,
        "notes": None,
        "molds": []
    })

    if comp.get("compound") is None and compound is not None:
        comp["compound"] = compound

    # ----------------------------
    # Inventory qtyByMoldSize
    # ----------------------------
    qty_by_mold_size = {}
    for s in size_cols:
        val = row.get(s)
        if pd.notna(val):
            qty_by_mold_size[s] = to_int_or_none(val)

    # ### NEW: ensure sizing keys exist with NULL values (Option A)
    # - Keys come from sizes observed in inventory for this component
    # - Values remain None until stakeholder fills them later
    sizing_comp = fam["sizingRulesByComponent"].setdefault(str(component_code), {
        "componentName": comp["displayName"],            # helpful for Excel export
        "soleType": str(sole_type),                      # helpful for Excel export
        "moldSizeToShoeSizes": {},                       # Option A dict
        "notes": "Auto-generated keys from inventory; values pending"
    })

    for mold_size in qty_by_mold_size.keys():
        # only create if missing; DO NOT overwrite if already filled later
        sizing_comp["moldSizeToShoeSizes"].setdefault(str(mold_size), None)

    # ----------------------------
    # Mold record
    # ----------------------------
    mold_record = {
        "vendorCode": vendor_code,
        "vendorName": vendor_name,
        "moldShopCode": mold_shop_code,
        "moldShopName": mold_shop_name,

        "location": {
            "code": location_code,
            "name": location
        },

        "initSeason": none_if_nan(row.get("season")),
        "lastDemandSeason": none_if_nan(row.get("last_demand_season")),

        "capacity": {
            "dailyOutput": to_float_or_none(row.get("daily_output")),
            "actualOutput": to_float_or_none(row.get("actual_output"))
        },

        "asset": {
            "moldCost": to_float_or_none(row.get("a_mold_cost")),
            "ownership": none_if_nan(row.get("mold_ownership")),
            "condition": none_if_nan(row.get("mold_condition")),
            "conditionNote": none_if_nan(row.get("mold_condition_note"))
        },

        "inventory": {
            "totalMoldQty": to_int_or_none(row.get("total_mold_qty")),
            "qtyByMoldSize": qty_by_mold_size
        },

        "comments": none_if_nan(row.get("comments")),
        "remark": none_if_nan(row.get("remark"))
    }

    comp["molds"].append(mold_record)

In [105]:
import json

with open("../data/mold_data.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False, default=str)